# Lab 1: profiling the NYC taxi data

Course 72ITDS40303 Big Data Analytics, Van Lang University.

This notebook is the working environment for Section 6 of the lab manual.
Run the cells in order. Compare every number with the manual.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("lab01-profile")
         .master("local[4]")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

RAW = "hdfs://localhost:9000/user/thaianh/nyc/raw"
jan = spark.read.parquet(f"{RAW}/yellow_tripdata_2024-01.parquet")

print("rows   ", f"{jan.count():,}")
print("columns", len(jan.columns))
jan.printSchema()

## Counting what is missing

Look carefully at the counts below before moving on.

In [ ]:
TOTAL = jan.count()

nulls = jan.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in jan.columns
]).collect()[0].asDict()

for col, n in sorted(nulls.items(), key=lambda kv: -kv[1]):
    if n > 0:
        print(f"{col:<24} {n:>9,}  {n/TOTAL:6.2%}")

## Proving it is one population, not five

Five columns missing the same number of values could be one group of rows,
or five unrelated groups of the same size. Those lead to opposite decisions,
so prove which it is rather than assuming.

In [ ]:
same = jan.filter(
    F.col("passenger_count").isNull() &
    F.col("RatecodeID").isNull() &
    F.col("store_and_fwd_flag").isNull() &
    F.col("congestion_surcharge").isNull() &
    F.col("Airport_fee").isNull()
).count()

print("rows where ALL FIVE are null:", f"{same:,}")

jan.groupBy("payment_type").agg(F.count("*").alias("rows")).orderBy("payment_type").show()

## Ranges

Print the minimum and maximum of every numeric and date column before doing
anything else. It takes one cell and finds problems no average will reveal.

In [ ]:
jan.select(
    F.min("tpep_pickup_datetime").alias("min_pickup"),
    F.max("tpep_pickup_datetime").alias("max_pickup"),
    F.min("fare_amount").alias("min_fare"),
    F.max("fare_amount").alias("max_fare"),
    F.min("trip_distance").alias("min_dist"),
    F.max("trip_distance").alias("max_dist"),
).show(truncate=False)